In [1]:
print("Hello World")

Hello World


In [2]:
import pandas as pd
import numpy as np
import os
import json

In [6]:
with open("../pages/model_info.json", "r") as af:
    model_info = json.load(af)

with open("../pages/models.json", "r") as m:
    models = json.load(m)
    
with open("../pages/results.json", "r") as r:
    results = json.load(r)
    
with open("../pages/domain_knowledge.json", "r") as d:
    domain_dict = json.load(d)

In [4]:
benchmark_metrics

{'Heart Disease Prediction': {'Accuracy': 91.02,
  'F1': 80.0,
  'Recall': 85.0,
  'Precision': 32.58,
  'ROC_AUC': 91.05,
  'False Positive Rate': 25.23},
 'Bone Fracture Detection': {'Accuracy': 72.0,
  'F1': 25.0,
  'Recall': 60.0,
  'Precision': 62.0,
  'ROC_AUC': 70.0,
  'False Positive Rate': 15.0}}

In [5]:
models

{'Classification': ['Heart Disease Prediction'],
 'Regression': [],
 'NLP': [],
 'CV': []}

In [6]:
outputs

{'Heart Disease Prediction': 'Classification'}

In [7]:
results

{'Heart Disease Prediction': {'07th Feb 2024': {'Production Data Summary': {'num_of_rows': 209,
    'no_of_rows_without_nan': 171,
    'no_of_rows_with_nan': 38,
    'completeness_score': 81.82,
    'uniqueness_score': 100.0,
    'no_of_rows_without_duplicates': 209,
    'no_of_rows_with_duplicates': 0,
    'validity_score': 74.16,
    'total_num_of_invalidrows': 54,
    'Metrics': {'Accuracy': 87.56,
     'Precision': 90.0,
     'Recall': 88.52,
     'F1': 89.26,
     'ROC_AUC': 87.37,
     'False Positive Rate': 13.79},
    'Drifting Metrics': ['Accuracy', 'ROC_AUC', 'False Positive Rate']},
   'Input Feature Details': {'Age': {'num_of_missing_values': 0,
     'uniqueness_score': 20.1,
     'num_of_datatype_mismatch': 0,
     'num_of_outliers': 0,
     'mean': 52.66,
     'median': 54.0,
     'std': 9.53,
     'min': 28,
     'max': 76,
     'iqr': 13.0,
     'type': 'int64',
     'ks-stat': 0.09,
     'p-value': 0.21,
     'status': False},
    'Sex': {'num_of_missing_values': 0,
  

### Create one single json file, that has all the info

In [8]:
temp_dict = dict()
target = "target"

In [9]:
temp_dict['Heart Disease Prediction'] = {"type": "Classification", "output_type": "Classification", "target": "target",
                                        "benchmark_metrics": benchmark_metrics['Heart Disease Prediction']}

In [10]:
temp_dict

{'Heart Disease Prediction': {'type': 'Classification',
  'output_type': 'Classification',
  'target': 'target',
  'benchmark_metrics': {'Accuracy': 91.02,
   'F1': 80.0,
   'Recall': 85.0,
   'Precision': 32.58,
   'ROC_AUC': 91.05,
   'False Positive Rate': 25.23}}}

In [11]:
model_dict = dict()

for i in temp_dict:
    
    model_dict[i] = temp_dict[i]['type']


In [12]:
model_dict

{'Heart Disease Prediction': 'Classification'}

In [13]:
with open("../pages/model_info.json", "w") as mf:
            json.dump(temp_dict, mf)

### Creating results_dict for CV models

In [14]:
from utils import determine_dtype_ft, ks_test, chi_test, classification_metrics, EncodeLabels, get_statistics

In [79]:
def get_results(model_name, date, ground_truth, production_data, baseline_data, model_type):
    
    cols = production_data.drop(["target"],axis=1).columns if model_type != "CV" else production_data.drop(["target", "paths"],axis=1).columns
    
    with open("../pages/benchmark.json", "r") as bench :
        benchmark_metrics = json.load(bench)
    
    categorical_ft, numerical_ft = determine_dtype_ft(baseline_data.drop(["target"],axis = 1))
    ks_results , _ = ks_test(baseline_data.drop(["target"],axis = 1),production_data.drop("target",axis=1),num_ft = numerical_ft)
    chi_results, _ = chi_test(baseline_data.drop(["target"],axis = 1),production_data.drop("target",axis=1),categorical_ft=categorical_ft)
    
    if model_type == "Classification" :
        gt, prod = EncodeLabels(ground_truth['target'],production_data['target'])
        metrics = classification_metrics(gt, prod, list(baseline_data['target'].unique()))
        pred_results, _ = chi_test(ground_truth['target'],production_data['target'], alpha = 0.05)
        
        performance = {
                "Metrics" : metrics ,  # check fpr 
                "Drifting Metrics" : [i for i in list(benchmark_metrics[model_name].keys()) if metrics[i] < benchmark_metrics[model_name][i]]
            }
    
    if model_type == "Regression":
        metrics = regression_metrics(ground_truth['target'],production_data['target'])
        pred_results, _ = ks_test(ground_truth['target'],production_data['target'], alpha = 0.05)
        
        performance = {
                "Metrics" : metrics ,  
                "Drifting Metrics" : [i for i in list(benchmark_metrics[model_name].keys()) if metrics[i] > benchmark_metrics[model_name][i]]
            }
    
    if model_type == "NLP":
        pass
    
    if model_type == "CV":
        
        # Performance Drift
        gt, prod = EncodeLabels(ground_truth['target'],production_data['target'])
        metrics = classification_metrics(gt, prod, list(baseline_data['target'].unique()))
        
        print(list(benchmark_metrics[model_name].keys()))
        
        performance = {
                "Metrics" : metrics ,  # check fpr 
                "Drifting Metrics" : [i for i in list(benchmark_metrics[model_name].keys()) if metrics[i] < benchmark_metrics[model_name][i]]
            }
        
        baseline_data.drop('paths', axis=1, inplace=True)
        production_data.drop('paths', axis=1, inplace=True)
        
        # Reading CSV with image information
        df = pd.read_csv(f"../pages/models/{model_name}/Production/{date}/production_paths_df.csv")
        
        # Prediction Drift Analysis
        pred_results, _ = chi_test(ground_truth['target'],production_data['target'], alpha = 0.05)


    if model_type != "CV":    
        
        completeness, _, WO_nan, w_nan, missing_data = data_completeness(production_data.drop(["target"],axis=1))
        uniqueness_score,u_s, _, _, no_of_rows_wo_dup = data_uniqueness(production_data.drop(["target"],axis=1))
        cat_mismatch, miss_dict, score,_, outliers_index, num_invalid = validity_check(baseline_data.drop("target",axis=1),production_data.drop(["target"],axis=1))
    
        
        summary = {
            "num_of_rows" : len(production_data),
            "no_of_rows_without_nan" : WO_nan,
            "no_of_rows_with_nan" : w_nan,
            "completeness_score" : completeness,
            "uniqueness_score" : uniqueness_score ,
            "no_of_rows_without_duplicates" : no_of_rows_wo_dup,
            "no_of_rows_with_duplicates" : len(production_data) - no_of_rows_wo_dup,
            "validity_score" : score,
            "total_num_of_invalidrows" : num_invalid

        }
        
        #     cols = production_data.drop(["probs","target"],axis=1).columns
        input_features = dict()
        input_u_s = {u_s['Feature'][i] : round(u_s['Value'][i],2) for i in range(len(u_s['Feature']))}
        datatype_mismatch = {i : len(cat_mismatch[i]) for i in cat_mismatch}
        outliers = {i : len(outliers_index[i]) for i in outliers_index}


        for i in cols:
            input_features[i] = {
                    "num_of_missing_values" : dict(missing_data)[i],
                    "uniqueness_score" : input_u_s[i],
                    "num_of_datatype_mismatch" : datatype_mismatch[i] if i in list(datatype_mismatch.keys()) else  0,
                    "num_of_outliers" : outliers[i],
                    }

            if i in categorical_ft:
                input_features[i].update(get_statistics(production_data[i], "categorical", chi_results))
            if i in numerical_ft:
                input_features[i].update(get_statistics(production_data[i], "numerical", ks_results))
    
    else:
        summary = {
            "Number of Images" : len(df),
            "Number of Unique Resolutions and Sizes found" : len(df['resolution'].unique()),
            "Unique Resolutions found" : df['resolution'].unique(),
            "Unique Sizes found": df['size'].unique(),
            "Statistics of Sharpness of all Images": {"Maximum": float(round(df['sharpness'].max(), 2)),
                                                     "Minimum": float(round(df['sharpness'].min(), 2)),
                                                     "Mean": float(round(df['sharpness'].mean(), 2))},
            "Statistics of Brightness of all Images": {"Maximum": float(round(df['brightness'].max(), 2)),
                                                     "Minimum": float(round(df['brightness'].min(), 2)),
                                                     "Mean": float(round(df['brightness'].mean(), 2))},
            "Statistics of Noise of all Images": {"Maximum": float(round(df['noise'].max(), 2)),
                                                     "Minimum": float(round(df['noise'].min(), 2)),
                                                     "Mean": float(round(df['noise'].mean(), 2))},
            "Number of Anomalies": float(np.sum(df['Anomaly'] == True))
        }
        
        input_features = dict()
        for i in cols:
            input_features[i] = get_statistics(production_data[i], "numerical", ks_results)

    summary.update(performance)
    
    prediction = {"prediction_drift" : pred_results}
    
    results_dict = {model_name : {date: {"Production Data Summary": summary, "Input Feature Details": input_features, "Output Feature Details": pred_results}}}
        
    
    return results_dict


In [80]:
baseline = pd.read_csv("../pages/models/Bone Fracture Detection/baseline.csv")

In [81]:
baseline.head()

,paths,target,contrast,energy,homogeneity,correlation,dissimilarity
0,./pages/models/Bone Fracture Detection/Baselin...,fracture,62.074968,0.181372,0.573962,0.981680,2.942447
1,./pages/models/Bone Fracture Detection/Baselin...,fracture,61.691944,0.183658,0.568558,0.982458,3.062721
2,./pages/models/Bone Fracture Detection/Baselin...,fracture,60.965311,0.189449,0.573245,0.983152,3.129324
3,./pages/models/Bone Fracture Detection/Baselin...,fracture,127.833353,0.172961,0.573006,0.959753,3.410453
4,./pages/models/Bone Fracture Detection/Baselin...,fracture,72.364583,0.173927,0.587243,0.977612,2.724903


In [82]:
baseline.columns

Index(['paths', 'target', 'contrast', 'energy', 'homogeneity', 'correlation',
       'dissimilarity'],
      dtype='object')

In [83]:
prod_run = pd.read_csv("../pages/models/Bone Fracture Detection/Production Runs/2024-03-01.csv").drop("Unnamed: 0", axis=1)

In [84]:
gt = pd.read_csv("../pages/models/Bone Fracture Detection/Ground Truths/2024-03-01.csv").drop("Unnamed: 0", axis=1)

In [85]:
prod_run.head()

,paths,contrast,energy,homogeneity,correlation,dissimilarity,probs,target
0,./pages/models/Bone Fracture Detection/Product...,61.754364,0.183457,0.570387,0.982428,3.054677,0.0,fractured
1,./pages/models/Bone Fracture Detection/Product...,90.351311,0.179828,0.569381,0.973787,3.361768,0.0,fractured
2,./pages/models/Bone Fracture Detection/Product...,302.318110,0.135699,0.499058,0.953433,5.731600,0.0,fractured
3,./pages/models/Bone Fracture Detection/Product...,234.358209,0.105948,0.514742,0.962378,4.386523,0.0,fractured
4,./pages/models/Bone Fracture Detection/Product...,243.981680,0.116100,0.552829,0.962515,4.081624,0.0,fractured


In [86]:
prod_run['contrast']

0       61.754364
1       90.351311
2      302.318110
3      234.358209
4      243.981680
          ...    
295    130.334942
296    129.038268
297    109.489986
298    104.305200
299    154.492077
Name: contrast, Length: 300, dtype: float64

In [87]:
gt.head()

,paths,target,contrast,energy,homogeneity,correlation,dissimilarity
0,./pages/models/Bone Fracture Detection/Product...,fractured,61.754364,0.183457,0.570387,0.982428,3.054677
1,./pages/models/Bone Fracture Detection/Product...,fractured,90.351311,0.179828,0.569381,0.973787,3.361768
2,./pages/models/Bone Fracture Detection/Product...,fractured,302.318110,0.135699,0.499058,0.953433,5.731600
3,./pages/models/Bone Fracture Detection/Product...,fractured,234.358209,0.105948,0.514742,0.962378,4.386523
4,./pages/models/Bone Fracture Detection/Product...,fractured,243.981680,0.116100,0.552829,0.962515,4.081624


In [88]:
results_dict_cv = get_results(model_name="Bone Fracture Detection", baseline_data=baseline, 
                              production_data=prod_run.drop("probs", axis=1),
                             ground_truth=gt, date="2024-03-01", model_type="CV")

['Accuracy', 'F1', 'Recall', 'Precision', 'ROC_AUC', 'False Positive Rate']


In [89]:
benchmark_metrics["Bone Fracture Detection"]

{'Accuracy': 72.0,
 'F1': 25.0,
 'Recall': 60.0,
 'Precision': 62.0,
 'ROC_AUC': 70.0,
 'False Positive Rate': 15.0}

In [78]:
results_dict_cv

{'Bone Fracture Detection': {'2024-03-01': {'Production Data Summary': {'Number of Images': 300,
    'Number of Unique Resolutions found': 7,
    'Unique Resolutions found': array(['(224, 224)', '(962, 1358)', '(1962, 1220)', '(1500, 1664)',
           '(750, 1256)', '(1066, 1652)', '(758, 1432)'], dtype=object),
    'Unique Sizes found': array([ 150528, 3919188, 7180920, 7488000, 2826000, 5283096, 3256368],
          dtype=int64),
    'Number of Unique Sizes found': 7,
    'Statistics of Sharpness of all Images': {'Maximum': 598.74,
     'Minimum': 29.78,
     'Mean': 156.25},
    'Statistics of Brightness of all Images': {'Maximum': 83.04,
     'Minimum': 30.7,
     'Mean': 53.47},
    'Statistics of Noise of all Images': {'Maximum': 77.44,
     'Minimum': 38.08,
     'Mean': 51.65},
    'Number of Anomalies': 40.0,
    'Metrics': {'report': {'fracture': {'precision': 0.7784810126582279,
       'recall': 0.8848920863309353,
       'f1-score': 0.8282828282828283,
       'support': 139

### Separating results dict

In [20]:
from utils import get_results

In [21]:
results

{'Heart Disease Prediction': {'07th Feb 2024': {'Production Data Summary': {'num_of_rows': 209,
    'no_of_rows_without_nan': 171,
    'no_of_rows_with_nan': 38,
    'completeness_score': 81.82,
    'uniqueness_score': 100.0,
    'no_of_rows_without_duplicates': 209,
    'no_of_rows_with_duplicates': 0,
    'validity_score': 74.16,
    'total_num_of_invalidrows': 54,
    'Metrics': {'Accuracy': 87.56,
     'Precision': 90.0,
     'Recall': 88.52,
     'F1': 89.26,
     'ROC_AUC': 87.37,
     'False Positive Rate': 13.79},
    'Drifting Metrics': ['Accuracy', 'ROC_AUC', 'False Positive Rate']},
   'Input Feature Details': {'Age': {'num_of_missing_values': 0,
     'uniqueness_score': 20.1,
     'num_of_datatype_mismatch': 0,
     'num_of_outliers': 0,
     'mean': 52.66,
     'median': 54.0,
     'std': 9.53,
     'min': 28,
     'max': 76,
     'iqr': 13.0,
     'type': 'int64',
     'ks-stat': 0.09,
     'p-value': 0.21,
     'status': False},
    'Sex': {'num_of_missing_values': 0,
  

In [ ]:
def separated_results(result_dict, model_name, model_type, prod_date, cols):
    
    results = result_dict[model_name][prod_date]
    
    
    input_details = results['Input Feature Details']
    
    # Performance Dict
    performance_dict = results['Metrics']
    
    # Data Drift Dict
    
    # Data Quality Dict
    
    # Prediction Dict
    
    pass

In [5]:
model_info['Heart']

{'Heart Disease Prediction': {'type': 'Classification',
  'output_type': 'Classification',
  'target': 'target',
  'benchmark_metrics': {'Accuracy': 91.02,
   'F1': 80.0,
   'Recall': 85.0,
   'Precision': 32.58,
   'ROC_AUC': 91.05,
   'False Positive Rate': 25.23}},
 'Heart Disease Classification': {'benchmark_metrics': {'Accuracy': 87.0,
   'F1': 59.0,
   'Recall': 82.0,
   'Precision': 67.0,
   'ROC_AUC': 50.0,
   'False Positive Rate': 42.0},
  'output_type': 'Classification',
  'type': 'Classification',
  'target': 'target',
  'domain_knowledge': '{"task": "Predict heart disease presence based on patient characteristics and test results. Data is collected from clinical assessments, medical tests, and patient history.", "column info": {"Age": {"data_type": "int64", "meaning": "Age of the patient in years, recorded at the time of assessment."}, "Sex": {"data_type": "object", "meaning": "Gender of the patient (M = Male, F = Female), recorded from patient demographics."}, "ChestPainT

In [7]:
results

{'Heart Disease Prediction': {'07th Feb 2024': {'Production Data Summary': {'num_of_rows': 209,
    'no_of_rows_without_nan': 171,
    'no_of_rows_with_nan': 38,
    'completeness_score': 81.82,
    'uniqueness_score': 100.0,
    'no_of_rows_without_duplicates': 209,
    'no_of_rows_with_duplicates': 0,
    'validity_score': 74.16,
    'total_num_of_invalidrows': 54,
    'Metrics': {'Accuracy': 87.56,
     'Precision': 90.0,
     'Recall': 88.52,
     'F1': 89.26,
     'ROC_AUC': 87.37,
     'False Positive Rate': 13.79},
    'Drifting Metrics': ['Accuracy', 'ROC_AUC', 'False Positive Rate']},
   'Input Feature Details': {'Age': {'num_of_missing_values': 0,
     'uniqueness_score': 20.1,
     'num_of_datatype_mismatch': 0,
     'num_of_outliers': 0,
     'mean': 52.66,
     'median': 54.0,
     'std': 9.53,
     'min': 28,
     'max': 76,
     'iqr': 13.0,
     'type': 'int64',
     'ks-stat': 0.09,
     'p-value': 0.21,
     'status': False},
    'Sex': {'num_of_missing_values': 0,
  

In [9]:
data_drift

NameError: name 'data_drift' is not defined